# CS5228-KDDM, 2025/26-2, Coursework 2
## CW2-1: Regression (3 marks)

#### Student Name: Lee Junyoung
#### Student Number: A0247530J

### Task: Regression using Linear Regression
Datasets: cs5228-Housing.csv 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# 1. Load dataset
df = pd.read_csv('cs5228-Housing.csv')


In [2]:
# first 5 rows of dataset
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,class_label
0,6510000,6670,3,1,3,yes,no,yes,no,no,0,yes,unfurnished,expensive
1,5880000,7160,3,1,1,yes,no,yes,no,no,2,yes,unfurnished,expensive
2,5873000,11460,3,1,3,yes,no,no,no,no,2,yes,semi-furnished,expensive
3,3430000,2610,3,1,2,yes,no,yes,no,no,0,yes,unfurnished,medium
4,3850000,7152,3,1,2,yes,no,no,no,yes,0,no,furnished,medium


In [3]:
# describe dataset
df.describe()

,price,area,bedrooms,bathrooms,stories,parking
count,5.450000e+02,545.000000,545.000000,545.000000,545.000000,545.000000
mean,4.766729e+06,5150.541284,2.965138,1.286239,1.805505,0.693578
std,1.870440e+06,2170.141023,0.738064,0.502470,0.867492,0.861586
min,1.750000e+06,1650.000000,1.000000,1.000000,1.000000,0.000000
25%,3.430000e+06,3600.000000,2.000000,1.000000,1.000000,0.000000
50%,4.340000e+06,4600.000000,3.000000,1.000000,2.000000,0.000000
75%,5.740000e+06,6360.000000,3.000000,2.000000,2.000000,1.000000
max,1.330000e+07,16200.000000,6.000000,4.000000,4.000000,3.000000


In [4]:

# Preprocessing: categorical to numerical
# Binary columns: yes=1, no=2
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 2})

# Furnishing status: avoid coding from zero
df['furnishingstatus'] = df['furnishingstatus'].map({'unfurnished': 1, 'semi-furnished': 2, 'furnished': 3})

# Define independent (X) and dependent (y) variables
X = df.drop(['price', 'class_label'], axis=1)
y = df['price']


### 1) Correlation Analysis and Feature Selection

We perform a correlation analysis to identify features that have a weak relationship with the target variable (`price`). Features with a correlation coefficient close to zero provide little predictive power and can potentially introduce noise into the model.

In [5]:

# Correlation analysis
correlation = X.corrwith(y)
print("Correlation analysis with price:\n", correlation)

# Identify features to discard
threshold = 0.1
discard_cols = correlation[abs(correlation) < threshold].index.tolist()
print(f"\nFeatures with correlation < {threshold}: {discard_cols}")


Correlation analysis with price:
 area                0.535997
bedrooms            0.366494
bathrooms           0.517545
stories             0.420712
mainroad           -0.296898
guestroom          -0.255517
basement           -0.187057
hotwaterheating    -0.093073
airconditioning    -0.452954
parking             0.384394
prefarea           -0.329777
furnishingstatus    0.304721
dtype: float64

Features with correlation < 0.1: ['hotwaterheating']


The feature `hotwaterheating` shows a correlation magnitude of approximately 0.09 with `price`, which is below our threshold of 0.1. This suggests that the presence of hot water heating has a negligible linear relationship with housing prices in this dataset. Discarding this feature helps simplify the model and may reduce the risk of overfitting to noise.

In [6]:
# Drop discardable features
X_filtered = X.drop(columns=discard_cols)
print(f"Remaining features: {X_filtered.columns.tolist()}")


Remaining features: ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']


In [7]:

# Split 90/10 with random_state=110
X_train, X_test, y_train, y_test = train_test_split(X_filtered, y, test_size=0.1, random_state=110)
# Check sizes of training and testing sets
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (490, 11), y_train shape: (490,)
X_test shape: (55, 11), y_test shape: (55,)


### 2) Normalization Impact Analysis

We evaluate the performance of standard Linear Regression both with and without data normalization (StandardScaler). This allows us to observe how feature scaling affects the model's coefficients and error metrics (MSE and MAE) for both training and testing sets.

In [8]:
def evaluate_regression(X_tr, y_tr, X_te, y_te, description):
    model = LinearRegression()
    model.fit(X_tr, y_tr)
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    mse_tr = mean_squared_error(y_tr, y_tr_pred)
    mae_tr = mean_absolute_error(y_tr, y_tr_pred)
    mse_te = mean_squared_error(y_te, y_te_pred)
    mae_te = mean_absolute_error(y_te, y_te_pred)
    
    print(f"--- {description} ---")
    print(f"Train MSE: {mse_tr:.2f}, MAE: {mae_tr:.2f}")
    print(f"Test  MSE: {mse_te:.2f}, MAE: {mae_te:.2f}\n")
    return mse_te, mae_te

# Case 1: Without Normalization
evaluate_regression(X_train, y_train, X_test, y_test, "Linear Regression (No Normalization)")

# Case 2: With Normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

baseline_mse, baseline_mae = evaluate_regression(X_train_scaled, y_train, X_test_scaled, y_test, "Linear Regression (With Normalization)")


--- Linear Regression (No Normalization) ---
Train MSE: 1196713817674.63, MAE: 799378.89
Test  MSE: 757562062965.61, MAE: 680604.97

--- Linear Regression (With Normalization) ---
Train MSE: 1196713817674.63, MAE: 799378.89
Test  MSE: 757562062965.61, MAE: 680604.97



### 3) Hyperparameter Optimization and K-Fold Cross Validation

To improve performance, we use **Ridge Regression**, which adds an L2 regularization penalty to the loss function to prevent overfitting. We employ **GridSearchCV** with **5-Fold Cross-Validation** to find the optimal hyperparameter `alpha` (the regularization strength).

In [9]:
# Define the model and parameter grid
ridge = Ridge()
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100, 1000]}

# Setup K-Fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=110)

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(ridge, param_grid, cv=kf, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)

best_ridge = grid_search.best_estimator_
print(f"Best Alpha: {grid_search.best_params_['alpha']}")

# Evaluate on test set
y_pred_opt = best_ridge.predict(X_test_scaled)
opt_mse = mean_squared_error(y_test, y_pred_opt)
opt_mae = mean_absolute_error(y_test, y_pred_opt)

print(f"Optimized Results (Ridge Regression):\nTest MSE: {opt_mse:.2f}, MAE: {opt_mae:.2f}")


Best Alpha: 10
Optimized Results (Ridge Regression):
Test MSE: 750876643046.24, MAE: 678194.15


### 4) Performance Comparison

The following table compares the baseline Linear Regression results (with normalization) with the optimized Ridge Regression results.

In [10]:
comparison_df = pd.DataFrame({
    'Metric': ['Test MSE', 'Test MAE'],
    'Baseline (Linear)': [baseline_mse, baseline_mae],
    'Optimized (Ridge)': [opt_mse, opt_mae]
})
comparison_df['Improvement (%)'] = (comparison_df['Baseline (Linear)'] - comparison_df['Optimized (Ridge)']) / comparison_df['Baseline (Linear)'] * 100

print("Final Comparison Table:")
print(comparison_df.to_string(index=False))


Final Comparison Table:
  Metric  Baseline (Linear)  Optimized (Ridge)  Improvement (%)
Test MSE       7.575621e+11       7.508766e+11         0.882491
Test MAE       6.806050e+05       6.781942e+05         0.354216
